## Ingesting the documents

In [10]:
from langchain_community.document_loaders.csv_loader import CSVLoader

vrmuseum_docs = CSVLoader(
    file_path='../../data/processed/vrmuseum.csv',
    content_columns=['content'],
    metadata_columns=['title', 'specimen_name'],
    encoding='utf-8'
).load()

In [14]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

model_name = 'Qwen/Qwen3-Embedding-0.6B'
embeddings = HuggingFaceEmbeddings(model_name=model_name)

In [15]:
import os
import chromadb
from langchain_chroma.vectorstores import Chroma

chroma_client = chromadb.CloudClient(
    tenant='b128d194-488a-4736-99d1-57f71009997d',
    database='vrmuseum',
    api_key='ck-7ADv2hRtVkSx3nk6n7mBn81GZZj5X9nCQAnGZEcZoy5a'
)

vector_store = Chroma(
    collection_name='specimens', 
    embedding_function=embeddings,
    client=chroma_client
)

In [17]:
from uuid import uuid4

def split_list(input_list, chunk_size=5461):
    for i in range(0, len(input_list), chunk_size):
        yield input_list[i:i + chunk_size]

for batch_docs in split_list(vrmuseum_docs):
    batch_ids = [str(uuid4()) for _ in range(len(vrmuseum_docs))]
    vector_store.add_documents(documents=batch_docs, ids=batch_ids)

In [18]:
dense_retriever = vector_store.as_retriever(search_kwargs={'k': 5})

In [21]:
# Test retrieval of five results
results = dense_retriever.invoke('preservation')

for doc in results:
    print(doc.metadata['specimen_name'])

RM 2727.2 ibis mummy fake
RM 5255 frozen caiman
RM 2727.1 ibis mummy
RM 2726.2 falcon mummy
RM 2728.1 cat mummy
